In [1]:
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

# ─────────────────────────────────────────────────────────────
# Read Coordinates from INP File
# ─────────────────────────────────────────────────────────────
def read_coordinates(inp_file):
    coords = {}
    with open(inp_file, 'r') as file:
        lines = file.readlines()

    in_coords = False
    for line in lines:
        line = line.strip()
        if line.upper().startswith('[COORDINATES]'):
            in_coords = True
            continue
        if in_coords and line.startswith('['):
            break
        if in_coords and line and not line.startswith(';'):
            parts = line.split()
            if len(parts) >= 3:
                coords[parts[0]] = (float(parts[1]), float(parts[2]))
    return coords

# ─────────────────────────────────────────────────────────────
# Read PUMP to NODE mapping from INP File
# ─────────────────────────────────────────────────────────────
def read_pump_locations(inp_file):
    pump_nodes = {}
    with open(inp_file, 'r') as file:
        lines = file.readlines()

    in_pumps = False
    for line in lines:
        line = line.strip()
        if line.upper().startswith('[PUMPS]'):
            in_pumps = True
            continue
        if in_pumps and line.startswith('['):
            break
        if in_pumps and line and not line.startswith(';'):
            parts = line.split()
            if len(parts) >= 3:
                pump_name = parts[0]
                inlet_node = parts[1]
                pump_nodes[pump_name] = inlet_node
    return pump_nodes

# ─────────────────────────────────────────────────────────────
# Plot One Map: Pumps Colored by Power Loss Category
# ─────────────────────────────────────────────────────────────
def plot_single_pump_powerloss_map(inp_file, html_export_path):
    coords = read_coordinates(inp_file)
    pump_to_node = read_pump_locations(inp_file)

    fig = go.Figure()

    # All nodes background
    x_all, y_all = zip(*[coords[n] for n in coords])
    fig.add_trace(go.Scatter(
        x=x_all, y=y_all,
        mode='markers',
        marker=dict(color='lightgrey', size=4),
        name='All Nodes',
        hoverinfo='skip'
    ))

    try:
        pumps_path = Path.home() / "Desktop" / "pumps_35-45.csv"
        pumps_df = pd.read_csv(pumps_path)

        # Filter to pumps that exist in the INP file
        pumps_df = pumps_df[pumps_df["Name"].isin(pump_to_node.keys())].copy()
        pumps_df["Node"] = pumps_df["Name"].map(pump_to_node)
        pumps_df = pumps_df[pumps_df["Node"].isin(coords.keys())].copy()

        # Bin setup
        bins = [0.0, 0.2, 0.4, 0.6, 0.8, 1.01]
        labels = ["0.0–0.2", "0.2–0.4", "0.4–0.6", "0.6–0.8", "0.8–1.0"]
        colors = ['green', 'yellowgreen', 'orange', 'darkorange', 'red']
        pumps_df["Category"] = pd.cut(pumps_df["pumps_lose_power"], bins=bins, labels=labels, include_lowest=True)

        print("\n📋 Pumps plotted on map, grouped by power loss category:")
        for label, color in zip(labels, colors):
            sub_df = pumps_df[pumps_df["Category"] == label]
            if not sub_df.empty:
                pump_coords = [coords[node] for node in sub_df["Node"]]
                x_pumps, y_pumps = zip(*pump_coords)
                fig.add_trace(go.Scatter(
                    x=x_pumps, y=y_pumps,
                    mode='markers',
                    marker=dict(size=10, color=color, symbol='triangle-up'),
                    name=f"Pumps Power Loss: {label}",
                    hoverinfo='text',
                    text=[f"{row['Name']}: {row['pumps_lose_power']:.2f}" for _, row in sub_df.iterrows()]
                ))

                # Print pump names in each category
                print(f"  {label} ({len(sub_df)} pumps):")
                for name in sub_df["Name"]:
                    print(f"    - {name}")

    except Exception as e:
        print(f"⚠️ Failed to read or map pump data: {e}")

    fig.update_layout(
        title="Pumps Power Loss Map",
        xaxis_title="X",
        yaxis_title="Y",
        height=700,
        width=1000,
        hovermode="closest"
    )

    fig.write_html(str(html_export_path))
    print(f"\n✅ Map saved to: {html_export_path}")

# ─────────────────────────────────────────────────────────────
# Run for Single INP File
# ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    inp_path = Path(r"C:\Users\csofios\Desktop\LongTermCSO\SWMM\GLWA_Core_v4_R05_2020_p2_harris_no_event_2.inp")
    html_output = Path(r"C:\Users\csofios\Desktop\LongTermCSO\SWMM\pumps_power_loss_map_35-45.html")
    plot_single_pump_powerloss_map(inp_path, html_output)



📋 Pumps plotted on map, grouped by power loss category:
  0.0–0.2 (8 pumps):
    - WRRF_PS2_P10
    - WRRF_PS2_P11
    - WRRF_PS2_P12
    - WRRF_PS2_P13
    - WRRF_PS2_P14
    - WRRF_PS2_P15
    - WRRF_PS2_P16
    - WRRF_PS2_P9
  0.2–0.4 (10 pumps):
    - MDOT_PumpI75Grand
    - MDOT_PumpI75GrandRiver
    - OakwoodPump1
    - OakwoodPump2
    - OakwoodPump3
    - OakwoodPump4
    - OakwoodPump5
    - OakwoodPump6
    - OakwoodPump7
    - OakwoodSanPump
  0.4–0.6 (20 pumps):
    - 4000_OAK
    - BH_SanPump1
    - BH_SanPump2
    - BH_StormPump1
    - BH_StormPump2
    - BH_StormPump3
    - BH_StormPump4
    - Kerby1
    - Kerby2
    - Kerby3
    - Kerby4
    - MDOT_PumpI75Ferry
    - MDOT_PumpI75I375
    - MDOT_PumpM10Edison
    - MDOT_PumpM10Eudid
    - MDOT_PumpM10Second
    - MDOT_PumpM8Hamilton
    - MDOT_PumpM8JosCampau
    - NESPS_Pump6_OMID
    - WRRF_PS1_P8
  0.6–0.8 (60 pumps):
    - BelleIsleStormPump2
    - CON-PMP-1
    - CON-PMP-10
    - CON-PMP-11
    - CON-PMP-12
    - C

In [2]:
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
from collections import defaultdict

# ─────────────────────────────────────────────────────────────
# Read Coordinates from INP File
# ─────────────────────────────────────────────────────────────
def read_coordinates(inp_file):
    coords = {}
    with open(inp_file, 'r') as file:
        lines = file.readlines()

    in_coords = False
    for line in lines:
        line = line.strip()
        if line.upper().startswith('[COORDINATES]'):
            in_coords = True
            continue
        if in_coords and line.startswith('['):
            break
        if in_coords and line and not line.startswith(';'):
            parts = line.split()
            if len(parts) >= 3:
                coords[parts[0]] = (float(parts[1]), float(parts[2]))
    return coords

# ─────────────────────────────────────────────────────────────
# Read PUMP to NODE mapping from INP File
# ─────────────────────────────────────────────────────────────
def read_pump_locations(inp_file):
    pump_nodes = {}
    with open(inp_file, 'r') as file:
        lines = file.readlines()

    in_pumps = False
    for line in lines:
        line = line.strip()
        if line.upper().startswith('[PUMPS]'):
            in_pumps = True
            continue
        if in_pumps and line.startswith('['):
            break
        if in_pumps and line and not line.startswith(';'):
            parts = line.split()
            if len(parts) >= 3:
                pump_name = parts[0]
                inlet_node = parts[1]
                pump_nodes[pump_name] = inlet_node
    return pump_nodes

# ─────────────────────────────────────────────────────────────
# Plot with grouped hover labels for overlapping pumps
# ─────────────────────────────────────────────────────────────
def plot_single_pump_powerloss_map(inp_file, html_export_path):
    coords = read_coordinates(inp_file)
    pump_to_node = read_pump_locations(inp_file)

    fig = go.Figure()

    # Background nodes
    x_all, y_all = zip(*[coords[n] for n in coords])
    fig.add_trace(go.Scatter(
        x=x_all, y=y_all,
        mode='markers',
        marker=dict(color='lightgrey', size=4),
        showlegend=False,
        #name='All Nodes',
        hoverinfo='skip'
    ))

    try:
        pumps_path = Path.home() / "Desktop" / "pumps_35-45.csv"
        pumps_df = pd.read_csv(pumps_path)

        # Filter pumps that exist in the model
        pumps_df = pumps_df[pumps_df["Name"].isin(pump_to_node.keys())].copy()
        pumps_df["Node"] = pumps_df["Name"].map(pump_to_node)
        pumps_df = pumps_df[pumps_df["Node"].isin(coords.keys())].copy()

        # Group by coordinates
        point_groups = defaultdict(list)
        for _, row in pumps_df.iterrows():
            node = row["Node"]
            if node in coords:
                key = coords[node]
                point_groups[key].append((row["Name"], row["pumps_lose_power"]))

        # Extract grouped hover labels
        x_vals, y_vals, hover_labels, color_vals = [], [], [], []

        for (x, y), pump_list in point_groups.items():
            x_vals.append(x)
            y_vals.append(y)
            color_vals.append(sum([v for _, v in pump_list]) / len(pump_list))  # avg power loss

            label_lines = [f"{name}: {val:.2f}" for name, val in pump_list]
            hover_labels.append("<br>".join(label_lines))

        # Plot
        fig.add_trace(go.Scatter(
            x=x_vals,
            y=y_vals,
            mode='markers',
            marker=dict(
                size=10,
                color=color_vals,
                colorscale='Viridis',
                cmin=0,
                cmax=1,
                colorbar=dict(title='Pumps Power Loss'),
                symbol='triangle-up',
                line=dict(width=1, color='black')
            ),
            text=hover_labels,
            hoverinfo='text',
            showlegend=False
            #name='Pumps'
        ))

        print(f"\n📋 Pumps plotted on map ({len(pumps_df)} total):")
        for name, loss in zip(pumps_df["Name"], pumps_df["pumps_lose_power"]):
            print(f"  - {name}: {loss:.2f}")

    except Exception as e:
        print(f"⚠️ Failed to read or map pump data: {e}")

    fig.update_layout(
        title="Pumps Power Loss Map (with Grouped Hover Info)",
        xaxis_title="X",
        yaxis_title="Y",
        height=700,
        width=1000,
        hovermode="closest",
        dragmode="zoom"
    )

    fig.write_html(str(html_export_path))
    print(f"\n✅ Map saved to: {html_export_path}")

# ─────────────────────────────────────────────────────────────
# Execute
# ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    inp_path = Path(r"C:\Users\csofios\Desktop\LongTermCSO\SWMM\GLWA_Core_v4_R05_2020_p2_harris_no_event_2.inp")
    html_output = Path(r"C:\Users\csofios\Desktop\LongTermCSO\SWMM\pumps_power_loss_map_35-45.html")
    plot_single_pump_powerloss_map(inp_path, html_output)



📋 Pumps plotted on map (272 total):
  - 4000_OAK: 0.57
  - 7Mile_DewateringPump: 0.95
  - 8MileRdPump#1: 0.95
  - 8MileRdPump#2: 0.95
  - 8MileRdPump#3: 0.95
  - 8MileRdPump#4: 0.95
  - 8MileRdPump#5: 0.97
  - AcaciaDummyPump: 0.99
  - BelleIsleSanPump: 0.98
  - BelleIsleStormPump1: 0.98
  - BelleIsleStormPump2: 0.74
  - BH_SanPump1: 0.50
  - BH_SanPump2: 0.46
  - BH_StormPump1: 0.41
  - BH_StormPump2: 0.41
  - BH_StormPump3: 0.41
  - BH_StormPump4: 0.59
  - BirminghamDummyPump: 0.89
  - BloomfieldDummyPump: 0.96
  - BonHeur1: 0.92
  - BonHeur2: 0.92
  - BonHeur3: 0.92
  - BonHeur4: 0.92
  - CC_Dewater1: 0.91
  - CC_Dewater2: 0.91
  - CC_Dewater3: 0.92
  - CC_Dewater4: 0.95
  - ChapatonIntermediateP2: 1.00
  - ChapatonIntermediateP3: 0.99
  - ChapatonSanitaryP1: 0.99
  - ChapatonStorm1: 0.99
  - ChapatonStorm2: 0.98
  - ChapatonStorm3: 0.82
  - CON-PMP-1: 0.66
  - CON-PMP-10: 0.66
  - CON-PMP-11: 0.66
  - CON-PMP-12: 0.66
  - CON-PMP-2: 0.66
  - CON-PMP-3: 0.66
  - CON-PMP-4: 0.66
  -

In [1]:
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

# ─────────────────────────────────────────────────────────────
# Read Coordinates from INP File
# ─────────────────────────────────────────────────────────────
def read_coordinates(inp_file):
    coords = {}
    with open(inp_file, 'r') as file:
        lines = file.readlines()

    in_coords = False
    for line in lines:
        line = line.strip()
        if line.upper().startswith('[COORDINATES]'):
            in_coords = True
            continue
        if in_coords and line.startswith('['):
            break
        if in_coords and line and not line.startswith(';'):
            parts = line.split()
            if len(parts) >= 3:
                coords[parts[0]] = (float(parts[1]), float(parts[2]))
    return coords

# ─────────────────────────────────────────────────────────────
# Plot nodes with color scale by flood level
# ─────────────────────────────────────────────────────────────
def plot_all_nodes_with_flood_scale(inp_file, flood_csv, html_export_path):
    coords = read_coordinates(inp_file)
    df = pd.read_csv(flood_csv)
    df.rename(columns=lambda x: x.strip(), inplace=True)

    df['Name'] = df['Name'].astype(str)
    df = df[df['Name'].isin(coords)]

    # Coordinates and flood values
    x_vals = [coords[n][0] for n in df['Name']]
    y_vals = [coords[n][1] for n in df['Name']]
    flood_vals = df['nodes_flooded'].astype(float)
    hover_texts = [f"Node: {n}<br>Flooded: {f}" for n, f in zip(df['Name'], flood_vals)]

    fig = go.Figure()

    # Add node points with color scale
    fig.add_trace(go.Scatter(
        x=x_vals,
        y=y_vals,
        mode='markers',
        marker=dict(
            size=4,
            color=flood_vals,
            colorscale='Reds',
            cmin=0,
            cmax=1,
            colorbar=dict(title='Flooded (0-1)'),
            line=dict(width=0.5, color='black')
        ),
        text=hover_texts,
        hoverinfo='text',
        name='All Nodes'
    ))

    fig.update_layout(
        title="All Nodes Colored by Basement Flooding (0 = No Flood, 1 = Flooded)",
        xaxis_title="X",
        yaxis_title="Y",
        height=700,
        width=900,
        hovermode="closest",
        dragmode="zoom"
    )

    fig.write_html(str(html_export_path))
    print(f"\n✅ Flood map with color scale saved to: {html_export_path}")
    print(f"🟡 Total nodes plotted: {len(df)}")

# ─────────────────────────────────────────────────────────────
# Execute
# ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    inp_path = Path(r"C:\Users\csofios\Desktop\LongTermCSO\SWMM\GLWA_Core_v4_R05_2020_p2_harris_no_event_2.inp")
    flood_csv_path = Path(r"C:\Users\csofios\Desktop\flooded_nodes_35-45_backup.csv")
    html_output = Path(r"C:\Users\csofios\Desktop\LongTermCSO\SWMM\all_nodes_floodscale_map_backup_35-45.html")

    plot_all_nodes_with_flood_scale(inp_path, flood_csv_path, html_output)



✅ Flood map with color scale saved to: C:\Users\csofios\Desktop\LongTermCSO\SWMM\all_nodes_floodscale_map_backup_35-45.html
🟡 Total nodes plotted: 18525
